# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and analyze the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, with all references by Croissant entity `@id` values for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
We'll load metadata and records from the Croissant schema using `mlcroissant`. This process parses all available record sets, fields, and columns so they can be referenced by their `@id` values.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review available record sets, fields, and their `@id`s as defined in the FAIR^2 Croissant schema.

We'll list all record set `@id`s present in the dataset, along with their field `@id`s (columns) for later use.

In [ ]:
# List all record set @id's, their name, and the field (column) @id's within each
print("Record sets in this dataset:")
record_sets_info = []

for rs in metadata.record_sets:
    print(f"- @id: {rs.id}")
    print(f"  name: {rs.name}")
    print("  fields (columns):")
    field_ids = []
    for field in rs.fields:
        print(f"    - @id: {field.id}, name: {field.name}, datatype: {field.data_type}")
        field_ids.append(field.id)
    record_sets_info.append({'id': rs.id, 'name': rs.name, 'fields': field_ids})
    print()

# Save the IDs for later use
record_set_ids = [r['id'] for r in record_sets_info]

## 3. Data Extraction
We'll extract data from a specific record set and load it into a pandas DataFrame for analysis.

Choose a record set by its `@id` from the overview above.

In [ ]:
# Select one record set for demonstration; use the first available one
if not record_set_ids:
    raise ValueError("No record sets found in metadata.")

selected_record_set_id = record_set_ids[0]
print(f"Using record set @id: {selected_record_set_id}\n")

dataframes = {}
for rid in record_set_ids:
    records = list(dataset.records(record_set=rid))
    if records:
        df = pd.DataFrame(records)
        dataframes[rid] = df
        print(f"{rid}: {df.shape[0]} records, {df.shape[1]} columns")
    else:
        print(f"{rid}: No records found.")

print("\nColumns in selected record set:")
if selected_record_set_id in dataframes:
    print(dataframes[selected_record_set_id].columns.tolist())
    dataframes[selected_record_set_id].head()
else:
    print(f"No DataFrame available for {selected_record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
We'll perform some basic EDA on a numeric field.

1. Filter records based on a numeric field chosen by its `@id`.
2. Normalize the field (z-score).
3. Optionally group by a categorical field (e.g., protein type or sample name), using their `@id` for reference.

In [ ]:
# Automatically pick a numeric field from the first record set
import numpy as np
rs = None
for r in metadata.record_sets:
    if r.id == selected_record_set_id:
        rs = r
        break

if rs is None:
    raise RuntimeError("Record set not found.")

# Find numeric field by data_type
numeric_field_id = None
for f in rs.fields:
    if f.data_type in ["schema:Integer", "schema:Float", "schema:Number"]:
        numeric_field_id = f.id
        break

if numeric_field_id is None:
    raise RuntimeError("No numeric field (Integer/Float/Number) found in selected record set.")
print(f"Selected numeric field @id: {numeric_field_id}")

df = dataframes[selected_record_set_id]

# Ensure correct conversion to numeric
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a string field
group_field_id = None
for f in rs.fields:
    if f.data_type == "schema:Text" and f.id != numeric_field_id and f.id in df.columns:
        group_field_id = f.id
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())
else:
    print("No suitable grouping field found.")

## 5. Visualization
We'll visualize the distribution of the selected numeric field after normalization, and the average value by group if a suitable grouping field exists.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id} (filtered)")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping field exists, barplot average by group
if group_field_id:
    top_groups = grouped_df.sort_values(numeric_field_id, ascending=False).head(10)
    plt.figure(figsize=(10, 4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=top_groups)
    plt.title(f"Top 10 {group_field_id} by mean {numeric_field_id}")
    plt.xticks(rotation=45, ha="right")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated loading the FAIR^2 mass spectrometry extracellular vesicle dataset using Croissant metadata, referenced all data structures by their `@id`, explored structure, extracted records, performed basic filtering and normalization, then visualized summary statistics for a selected numeric field.

Further analysis may include deeper biological questions, multi-field visualizations, or integration with downstream proteomics tools.